In [1]:
import scanpy as sc
import numpy as np
import scipy.sparse as sp
import logging
import rpy2.robjects as ro
import anndata2ri
from rpy2.robjects import r
from rpy2.robjects.conversion import localconverter
import rpy2.rinterface_lib.callbacks as rcb
from scipy.io import mmread
import os

rcb.logger.setLevel(logging.ERROR)


In [2]:
path_template = ("../results/03_star/{sample}.Solo.out/{output}/{kind}/")

days = [0, 2, 5, 7]

samples = [
    "D2_C",
    "D2_D",
    "D5_A",
    "D5_B",
    "D5_C",
    "D7_C",
    "D7_D",
    "D0_A",
    "D0_B"
]


In [3]:
os.makedirs("../results/adata", exist_ok=True)

## Function to run SoupX

In [4]:
def run_soupx(filtered, raw):
    # Get clusters
    tmp = filtered.copy()
    sc.pp.normalize_total(tmp)
    sc.pp.log1p(tmp)
    sc.pp.highly_variable_genes(tmp)
    sc.pp.pca(tmp)
    sc.pp.neighbors(tmp)
    sc.tl.leiden(tmp, flavor="igraph")
    clusters = tmp.obs["leiden"]
    
    # Get cell and gene ids
    cells = filtered.obs_names
    genes = filtered.var_names
    droplets = raw.obs_names

    # Create matrix
    mat_filtered = sp.csc_matrix(filtered.X.T)
    mat_raw = sp.csc_matrix(raw.X.T)

    with localconverter(anndata2ri.converter):
        # Pass objects into R
        ro.globalenv["mat_filtered"] = mat_filtered
        ro.globalenv["mat_raw"] = mat_raw
        ro.globalenv["genes"] = ro.StrVector(list(genes))
        ro.globalenv["cells"] = ro.StrVector(list(cells))
        ro.globalenv["droplets"] = ro.StrVector(list(droplets))
        ro.globalenv["clusters"] = ro.StrVector(list(clusters.astype(str)))

        r("""
        suppressMessages(library(SoupX))
        suppressMessages(library(Matrix))

        mat_filtered <- Matrix(mat_filtered, sparse=TRUE)
        rownames(mat_filtered) <- genes
        colnames(mat_filtered) <- cells

        mat_raw <- Matrix(mat_raw, sparse=TRUE)
        rownames(mat_raw) <- genes
        colnames(mat_raw) <- droplets

        names(clusters) <- cells
        soupx_sc <- SoupChannel(mat_raw, mat_filtered)
        soupx_sc <- setClusters(soupx_sc, clusters)
        soupx_sc <- autoEstCont(soupx_sc, doPlot=FALSE)
        out <- as.matrix(adjustCounts(soupx_sc))
        rho <- soupx_sc$metaData$rho
        """)

        out = ro.globalenv["out"]
        rho = ro.globalenv["rho"]        
        filtered.layers["soupx"] = np.array(out).T
        return rho


## Process Data
Loop over each sample, do ambient RNA correction with Soupx, and concatenate data

In [ ]:
adata_list = []
for sample in samples:
    print(f"Processing sample: {sample}")  
    
    adata_filtered = sc.read_10x_mtx(
        path=path_template.format(sample=sample, output="Gene", kind="filtered"), 
        cache=True)
    
    adata_raw = sc.read_10x_mtx(
        path=path_template.format(sample=sample, output="Gene", kind="raw"), 
        cache=True)

    velo_dir = path_template.format(sample=sample, output="Velocyto", kind="filtered")
    
    spliced = mmread(velo_dir + "spliced.mtx").T.tocsr()
    unspliced = mmread(velo_dir + "unspliced.mtx").T.tocsr()
    ambiguous = mmread(velo_dir + "ambiguous.mtx").T.tocsr()

    adata_filtered.layers["spliced"] = spliced
    adata_filtered.layers["unspliced"] = unspliced
    adata_filtered.layers["ambiguous"] = ambiguous

    # Create some meta data columns
    adata_filtered.obs["sample"] = sample
    adata_filtered.obs["day"] = int(sample.lstrip("D").split("_")[0])

    # Run soupx to compute corrected count matrix
    rho = run_soupx(adata_filtered, adata_raw)
    print(f"Estimated contamination fraction of {rho[0]} for sample {sample}")

    adata_list.append(adata_filtered)

Processing sample: D2_C
Estimated contamination fraction of 0.01 for sample D2_C
Processing sample: D2_D
Estimated contamination fraction of 0.023 for sample D2_D
Processing sample: D5_A


In [ ]:
# Concatenate
print("Concatenating data")

adata = sc.concat(adata_list)
adata.obs["sample"] = adata.obs["sample"].astype("category")
adata.obs["day"] = adata.obs["day"].astype("category").cat.reorder_categories(days)
 
# Create rounded soupx layer
adata.layers["soupx_rounded"] = np.round(adata.layers["soupx"])

# Store backup raw counts layer
adata.layers["raw_counts"] = adata.X

adata.obs_names_make_unique()

In [ ]:
adata.write("../results/adata/05-filter-ambient.h5ad")